In [2]:
from dotenv import load_dotenv
load_dotenv()

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI

model = ChatGoogleGenerativeAI(model = "gemini-3-flash-preview")
# model = ChatOpenAI(model = "gpt-5-nano")

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name ="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6999.68it/s]


In [7]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [10]:
import bs4

loader = WebBaseLoader(web_path=("https://www.ebsco.com/research-starters/biography/elon-musk",))

docs = loader.load()

docs

[Document(metadata={'source': 'https://www.ebsco.com/research-starters/biography/elon-musk', 'title': 'Elon Musk | Biography | Research Starters | EBSCO Research', 'description': "<p>Elon Musk is a South African-born entrepreneur known for his significant contributions to technology and space exploration, as well as his involvement in electric vehicles. Born on June 28, 1971, Musk emigrated to Canada in 1988 and later pursued education in business and physics at the University of Pennsylvania. He co-founded Zip2, an online content publishing company, which was sold for nearly $350 million, providing him with the capital to invest in various ventures. Musk gained prominence as a key figure in the development of PayPal, which was acquired by eBay, leading to substantial financial returns.</p>\n<p>He is best known as the CEO and founder of SpaceX, which has revolutionized space travel with projects like the Falcon 9 rocket and the Starlink satellite internet service, and as the driving fo

In [13]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splits=text_splitter.split_documents(docs)

vectorstore=Chroma.from_documents(documents=splits,embedding=embedding)

retriever=vectorstore.as_retriever()
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x139694b20>, search_kwargs={})

In [ ]:
## Prompt Template
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system",system_prompt),
        ("human", "{input}")
    ]
)

In [15]:
question_answer_chain = create_stuff_documents_chain(model, prompt)

rag_chain = create_retrieval_chain(retriever, question_answer_chain)


In [18]:
response = rag_chain.invoke({"input":"who is elon mask"})

response['answer']

'Elon Musk is a South African-born entrepreneur recognized for his significant contributions to technology, space exploration, and electric vehicles. He co-founded Zip2 and played a major role in the development of PayPal before later rebranding Twitter as "X." Musk studied business and physics at the University of Pennsylvania and has been a central figure in several high-profile tech ventures.'

In [21]:
response = rag_chain.invoke({"input":"tell about early career?"})

response['answer']

"I don't know. The provided text only covers Elon Musk's early life and his move to Canada, but it does not mention his early career."

## Adding chat history

In [ ]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder

contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)
contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system",contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human","{input}")
    ]
)

In [26]:
history_aware_retriever = create_history_aware_retriever(model, retriever, contextualize_q_prompt)
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x139694b20>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk'

In [27]:
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

In [28]:
question_answer_chain=create_stuff_documents_chain(model,qa_prompt)
rag_chain=create_retrieval_chain(history_aware_retriever,question_answer_chain)

In [32]:
from langchain_core.messages import AIMessage,HumanMessage
chat_history=[]
question="who is elon mask"
response1=rag_chain.invoke({"input":question,"chat_history":chat_history})


chat_history.extend(
    [
        HumanMessage(content=question),
        AIMessage(content=response1["answer"])
    ]
)

question2="Tell about early career"
response2=rag_chain.invoke({"input":question,"chat_history":chat_history})
print(response2['answer'])

Elon Musk is a South African-born entrepreneur known for his leadership of SpaceX and Tesla, Inc. He co-founded Zip2 and was a key figure in PayPal before acquiring and rebranding Twitter as X. Additionally, he has initiated innovative projects like Neuralink, which focuses on brain-computer interface technologies.


In [31]:
chat_history

[HumanMessage(content='who is elon mask', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Elon Musk is a South African-born entrepreneur recognized for his significant contributions to technology, space exploration, and electric vehicles. He co-founded Zip2 and was a key figure in the development of PayPal before acquiring and rebranding Twitter as "X." Musk studied business and physics at the University of Pennsylvania after emigrating to Canada in 1988.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [33]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory


store = {}


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)


/Users/rachin/Desktop/GenAI/genai/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [34]:
conversational_rag_chain.invoke(
    {"input": "who is elon mask?"},
    config={
        "configurable": {"session_id": "abc123"}
    },  # constructs a key "abc123" in `store`.
)["answer"]

'Elon Musk is a South African-born entrepreneur known for his significant contributions to technology, space exploration, and electric vehicles. He co-founded companies like Zip2 and was a key figure in the development of PayPal before later rebranding Twitter as "X." He holds degrees in business and physics from the University of Pennsylvania and has been a major figure in the path toward private spaceflight.'

In [35]:
conversational_rag_chain.invoke(
    {"input": "tell about early career?"},
    config={"configurable": {"session_id": "abc123"}},
)["answer"]

"Elon Musk's early career began after he studied business and physics at the University of Pennsylvania and co-founded Zip2, which sold for nearly $350 million. He then became a prominent figure in the development of PayPal, which provided him with significant capital after it was acquired by eBay. These early successes allowed him to invest in his later ventures, including SpaceX and Tesla."

In [36]:
store

{'abc123': InMemoryChatMessageHistory(messages=[HumanMessage(content='who is elon mask?', additional_kwargs={}, response_metadata={}), AIMessage(content='Elon Musk is a South African-born entrepreneur known for his significant contributions to technology, space exploration, and electric vehicles. He co-founded companies like Zip2 and was a key figure in the development of PayPal before later rebranding Twitter as "X." He holds degrees in business and physics from the University of Pennsylvania and has been a major figure in the path toward private spaceflight.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='tell about early career?', additional_kwargs={}, response_metadata={}), AIMessage(content="Elon Musk's early career began after he studied business and physics at the University of Pennsylvania and co-founded Zip2, which sold for nearly $350 million. He then became a prominent figure in the development of PayPal, which provi

In [38]:
conversational_rag_chain.invoke({
    "input":"what is his company name"},
    config = {"configurable":{"session_id":"abc12"}}
    )['answer']

'Elon Musk has co-founded and led several companies, including Zip2 and the online payment-processing company PayPal. According to the text, he is also the co-founder and CEO of Space Exploration Technologies (SpaceX) and Tesla Motors. These ventures focus on technology, space exploration, and electric vehicles.'

In [39]:
store

{'abc123': InMemoryChatMessageHistory(messages=[HumanMessage(content='who is elon mask?', additional_kwargs={}, response_metadata={}), AIMessage(content='Elon Musk is a South African-born entrepreneur known for his significant contributions to technology, space exploration, and electric vehicles. He co-founded companies like Zip2 and was a key figure in the development of PayPal before later rebranding Twitter as "X." He holds degrees in business and physics from the University of Pennsylvania and has been a major figure in the path toward private spaceflight.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='tell about early career?', additional_kwargs={}, response_metadata={}), AIMessage(content="Elon Musk's early career began after he studied business and physics at the University of Pennsylvania and co-founded Zip2, which sold for nearly $350 million. He then became a prominent figure in the development of PayPal, which provi